# Phase 2 — device-resident conv2d parity (CPU vs GPU)

`dconv2d` (im2col + `bk::gemm`, col2im scatter) in `pure/dtensor.hpp`.
L = sum(SiLU(conv2d(X,W,b))), backward, finite-difference grad-check.
**Runtime → GPU (T4)**, Run all. Both must print the same L/checksums and `grad-check OK`.


In [ ]:
!nvidia-smi -L
!nvcc --version | tail -1


In [ ]:
%cd /content
!rm -rf yolov8_cpp
!git clone -q --branch main https://github.com/yomei-o/yolov8_cpp.git
%cd /content/yolov8_cpp


### GPU (nvcc, needs -DUSE_CUDA + -arch=native)


In [ ]:
!nvcc -x cu -O2 -std=c++17 --extended-lambda -arch=native -DUSE_CUDA pure/dconv_test.cpp -o dconv_gpu
!./dconv_gpu; echo "exit=$?"


### CPU (g++, same source)


In [ ]:
!g++ -O2 -std=c++17 -I/usr/local/cuda/include -I/usr/local/cuda/include/cccl \
     -DTHRUST_DEVICE_SYSTEM=THRUST_DEVICE_SYSTEM_CPP pure/dconv_test.cpp -o dconv_cpu
!./dconv_cpu; echo "exit=$?"


---
**Expected** (matches local MSVC CPU): `L = 60.257778`, `dX=142.763622 dW=55.574032 dB=39.338148`, `grad-check OK` on both (~1e-3 float tol).
